<a href="https://colab.research.google.com/github/matchapresso/AI-Agent-with-RAG/blob/main/1.%20End-to-end%20RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.Executive Summary
This project implements a robust Retrieval-Augmented Generation (RAG) pipeline designed to answer technical questions based on the seminal paper "Attention Is All You Need" (Vaswani et al., 2017).

The solution moves beyond a basic tutorial setup by incorporating Hybrid Retrieval (combining keyword precision with semantic understanding) and preparing the architecture for Agentic Workflows.

## Key Technical Decisions:

* **Model Selection**: Switched to Qwen 2.5 (7B-Instruct). This model was chosen over Llama 3.1 because it is Apache 2.0 licensed (simplifying deployment) and demonstrates superior performance in "tool use" and strict adherence to system instructions, which is critical for future agent capabilities.

* **Hybrid Retrieval**: Implemented an Ensemble Retriever merging ChromaDB (Dense Vector Search) and BM25 (Sparse Keyword Search). This ensures accurate retrieval for both specific hyperparameter queries (e.g., "d_model=512") and high-level conceptual questions (e.g., "Why self-attention?").

* **Optimization**: The pipeline is optimized for T4 GPUs (16GB VRAM), utilizing 4-bit NF4 quantization via bitsandbytes to reduce memory footprint from ~15GB to ~5GB while maintaining inference quality.

In [ ]:
# INITIAL SETUP (Includes Arxiv + BM25)
import os
import signal

print("⏳ Installing libraries...")

# 1. Base Compatibility (Prevents crashes)
os.system('pip install "numpy<2.0"')

# 2. Main AI Libraries (New bitsandbytes for 2025)
os.system('pip install -U "bitsandbytes>=0.45.0" transformers accelerate peft triton')
os.system('pip install --no-cache-dir --force-reinstall spacy thinc')

# 3. RAG Tools (LangChain + Arxiv + BM25 Engine)
# Added 'arxiv' and 'pymupdf' for the Loader
# Added 'rank_bm25' for the Retriever
os.system('pip install -qU langchain langchain-community langchain-classic langchain-core langchain-huggingface langchain-chroma')
os.system('pip install -qU sentence-transformers rank_bm25 pypdf chromadb huggingface_hub arxiv pymupdf')

print("✅ Installation complete. Restarting kernel...")
os.kill(os.getpid(), signal.SIGKILL)

⏳ Installing libraries...


# 2.Technical Implementation Breakdown


## Step 1: Environment Setup & Dependency Management

**Context:** In pre-built environments like Kaggle/Colab, library version conflicts are common. This cell handles the "dependency hell" upfront.

* **Key Fixes**:
  1. `numpy<2.0`: Downgraded NumPy to prevent binary incompatibility errors with spacy and thinc (a common issue in 2025/2026 Python stacks).
  2. `bitsandbytes>=0.45.0`: Forced the latest version to support the newer Triton 3.0+ hardware drivers present on the GPU.
  3.`arxiv` & `pymupdf`: Installed lightweight tools to scrape and parse PDFs directly from the Arxiv source, removing the need for manual file uploads.

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token="INSERT_TOKEN_HERE") #HuggingFace Token

In [ ]:
import os
import torch
import warnings
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_community.document_loaders import PyPDFLoader, OnlinePDFLoader, ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import ArxivLoader
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline

## Step 2: Model Initialization (Qwen 2.5 7B)
**Context**: Loading the reasoning engine.

**Logic:**
* **Model**: `Qwen/Qwen2.5-7B-Instruct`. Chosen for its massive 32k+ context window and strong reasoning performance.

* **Quantization**: `BitsAndBytesConfig` (NF4) is used to compress the model weights. This allows a 7B parameter model to run purely on the GPU without offloading to the (slower) CPU.

* **Temperature**: Set to 0.1. Low temperature is crucial for RAG to ensure the model acts as a factual retrieval engine rather than a creative writer.

In [ ]:
def setup_kaggle_rag():
    print("⚙️ Initializing Qwen 2.5 (7B) Pipeline...")

    # 1. Embeddings (Qwen works best with BGE or its own embeddings, but BGE is safer)
    print("🧠 Loading Embeddings...")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-base-en-v1.5",
        model_kwargs={'device': 'cuda'}
    )

    # 2. Model Configuration
    # Qwen 2.5 7B is powerful but fits on T4 GPU with 4-bit quantization
    model_id = "Qwen/Qwen2.5-7B-Instruct"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"⬇️ Loading {model_id} (No Token Needed)...")

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
    except Exception as e:
        print(f"❌ Error loading Qwen: {e}")
        raise e

    # 3. Create Pipeline
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=1024,      # Qwen handles long outputs well
        temperature=0.1,          # Low temp for precise RAG
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )

    llm = HuggingFacePipeline(pipeline=pipe)

    print("Qwen 2.5 Ready! (Best for Agents & RAG)")
    return embeddings, llm

## Step 3: Data Processing & Hybrid Indexing (ETL)
**Context:** The "Retrieval" backbone of the system.

**Process:**

* **Extract:** Uses `ArxivLoader` to download paper ID `1706.03762`.

* **Transform**: Uses `RecursiveCharacterTextSplitter (chunk_size=512)` to break text into logical paragraphs with 50-token overlap to preserve context across boundaries.

* **Load (Hybrid)**:

* **Dense Index (ChromaDB)**: vectorizes text for semantic search (understanding meaning).

* **Sparse Index `(BM25)`**: indexes keywords for exact matching (understanding terms).

* **Ensemble**: Combines both retrievers with a 50/50 weight.

In [ ]:
def process_data(arxiv_id, embeddings):
    print(f"📄 Loading Arxiv Paper ID: {arxiv_id}")

    # 1. Load Data using ArxivLoader (No complex dependencies!)
    # load_max_docs=1 ensures we only get the specific paper we asked for
    loader = ArxivLoader(query=arxiv_id, load_max_docs=1)

    # "get_summaries_as_docs=False" downloads the FULL PDF text, not just the abstract
    raw_docs = loader.load()

    # Check if we actually got data
    if not raw_docs:
        raise ValueError("❌ Could not find paper. Check your Arxiv ID.")

    print(f"   - Title: {raw_docs[0].metadata.get('Title', 'Unknown')}")
    print(f"   - Pages: {len(raw_docs[0].page_content) // 1000}k characters")

    # 2. Split (Same as before)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,
        chunk_overlap=50,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = text_splitter.split_documents(raw_docs)

    # 3. Add Metadata
    for i, chunk in enumerate(chunks):
        chunk.metadata["source"] = f"arxiv_{arxiv_id}"
        chunk.metadata["id"] = i

    # 4. Index
    print("🔎 Building Vector Index...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="kaggle_arxiv_test",
        persist_directory="/kaggle/working/chroma_db"
    )

    # 5. Hybrid Retrieval
    dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 3

    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[0.5, 0.5]
    )

    return ensemble_retriever

## Step 4: Pipeline Orchestration
**Context:** The integration layer.

**Logic:** This function ties the Model (Step 3) and Data (Step 4) together. It returns the chain, LLM, and retriever objects globally so they can be reused for the evaluation steps without reloading the model (which would be slow).

In [ ]:
def run_rag_system():
    # Setup
    embeddings, llm = setup_kaggle_rag()

    # ✅ CHANGED: Use Arxiv ID instead of URL
    # "1706.03762" = Attention Is All You Need
    # "2307.09288" = Llama 2 Paper
    arxiv_id = "1706.03762"

    retriever = process_data(arxiv_id, embeddings)

    # Prompt
    template = """You are an AI researcher. Answer based ONLY on the context.
    Context: {context}
    Question: {question}
    Answer:"""
    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # Chain
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # Test
    query = "Why is self-attention useful?"
    print(f"\n❓ Question: {query}")
    result = chain.invoke(query)
    print(f"💡 Answer: {result}")

    return chain, llm, retriever

# Run it
# We unpack the results into global variables here
final_chain, global_llm, global_retriever = run_rag_system()

print("\n✅ Success! 'global_retriever' is now available for the evaluation steps.")

## Step 5: Prompt Engineering (The "G" in RAG)
**Context:** satisfying Requirement #4 ("Optimize the prompt").

* **Strategy:** I implemented a structured System Prompt.

* **Persona:** "Senior Research Assistant" (sets a formal tone).

* **Constraints:** "Answer strictly based on the context" (reduces hallucinations).

* **Chain-of-Thought (CoT):** "Think step-by-step" (forces the model to reason before outputting the final answer, significantly improving accuracy on complex queries).

In [ ]:
# ==========================================
# REQ #4: FINAL RAG PIPELINE
# ==========================================

def build_optimized_rag(retriever, llm):
    # ---------------------------------------------------------
    # PROMPT OPTIMIZATION (Best Practice)
    # ---------------------------------------------------------
    # 1. Persona: "Senior Research Assistant" sets the tone.
    # 2. Constraints: "Strictly based on context" prevents hallucinations.
    # 3. Chain of Thought: "Think step-by-step" improves reasoning.
    # 4. Fallback: Explicit instruction for missing info.
    template = """You are a Senior Research Assistant. Your goal is to answer technical questions accurately using the provided paper segments.

    INSTRUCTIONS:
    1. Read the Context below carefully.
    2. Think step-by-step: Does the Context contain the answer?
    3. If YES: Answer concisely and cite the section if possible.
    4. If NO: Say "I cannot find this information in the provided context."

    CONTEXT:
    {context}

    QUESTION:
    {question}

    ANSWER (Step-by-Step):"""

    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        # Adds "Source" metadata to the context so the LLM knows where info comes from
        return "\n\n".join(f"[Source: Page {d.metadata.get('page', '0')}] {d.page_content}" for d in docs)

    rag_chain = (
        {"context": global_retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | global_llm
        | StrOutputParser()
    )

    return rag_chain

# Initialize
# Assuming 'retriever' and 'llm' are already created from previous steps
rag_chain = build_optimized_rag(global_retriever, global_llm)
print("✅ Optimized RAG Pipeline Ready.")

## Step 6: Evaluation Dataset
**Context:** Requirement #5 (20 Questions).

* **Design**: The dataset is split into two categories to test different capabilities:

* **Factual Retrieval:** "What is `d_model`?" (Tests BM25/Exact Match).

* **Conceptual Reasoning:** "Why is self-attention faster?" (Tests Vector Search/Semantic Understanding).

In [ ]:
# ==========================================
# REQ #5: EVALUATION DATASET (20 Q&A Pairs)
# ==========================================
eval_dataset = [
    # --- FACTUAL RETRIEVAL ---
    {"question": "What is the dominant sequence transduction model based on?", "answer": "Complex recurrent or convolutional neural networks", "key_term": "recurrent"},
    {"question": "What mechanism does the Transformer use instead of recurrence?", "answer": "Attention mechanisms (Self-Attention)", "key_term": "attention"},
    {"question": "What is the BLEU score achieved on the WMT 2014 English-to-German task?", "answer": "28.4", "key_term": "28.4"},
    {"question": "How many encoder layers are in the base model?", "answer": "6", "key_term": "6"},
    {"question": "What is the dimension of the model (d_model)?", "answer": "512", "key_term": "512"},
    {"question": "How many attention heads (h) are used in the base model?", "answer": "8", "key_term": "8"},
    {"question": "What optimizer is used for training?", "answer": "Adam optimizer", "key_term": "Adam"},
    {"question": "What regularization technique is used during training?", "answer": "Residual Dropout / Label Smoothing", "key_term": "dropout"},
    {"question": "What is the computational complexity of Self-Attention per layer?", "answer": "O(1)", "key_term": "O(1)"},
    {"question": "What hardware was used to train the models?", "answer": "8 P100 GPUs", "key_term": "P100"},

    # --- CONCEPTUAL / REASONING ---
    {"question": "Why is self-attention faster than recurrent layers for long sequences?", "answer": "It allows parallelization and has shorter path lengths between positions.", "key_term": "parallelization"},
    {"question": "What is the purpose of positional encodings?", "answer": "To inject information about the relative or absolute position of tokens since the model has no recurrence.", "key_term": "position"},
    {"question": "Explain the architecture of the Encoder.", "answer": "Stack of 6 identical layers, each with two sub-layers: multi-head self-attention and feed-forward network.", "key_term": "sub-layers"},
    {"question": "Explain the architecture of the Decoder.", "answer": "Stack of 6 identical layers with an extra sub-layer (masked attention) to prevent peeping into the future.", "key_term": "masked"},
    {"question": "What is 'Scaled Dot-Product Attention'?", "answer": "It computes the dot products of query and keys, divides by square root of dk, and applies softmax.", "key_term": "softmax"},
    {"question": "Why is the dot product scaled by 1/sqrt(dk)?", "answer": "To counteract the effect of large dot products pushing softmax into regions with extremely small gradients.", "key_term": "gradients"},
    {"question": "What is Multi-Head Attention?", "answer": "Linearly projecting queries, keys, and values h times and performing attention in parallel.", "key_term": "projecting"},
    {"question": "What label smoothing value was used?", "answer": "0.1", "key_term": "0.1"},
    {"question": "How does the Transformer reduce sequential computation?", "answer": "By relying entirely on attention mechanisms, allowing global dependencies to be drawn in constant time.", "key_term": "sequential"},
    {"question": "What is the FFN equation?", "answer": "FFN(x) = max(0, xW1 + b1)W2 + b2", "key_term": "max"}
]

## Step 7: Evaluation Engine
**Context:** Requirement #6 (Metrics).

Metrics Implemented:

* **1.Hit Rate @ `k`:** Did the retrieval system find the correct document?

* **MRR (Mean Reciprocal Rank)**: Was the relevant document ranked at the top?

* **Faithfulness Proxy:** A keyword-based check to verify if the LLM's answer contained the critical information from the ground truth.

In [ ]:
# ==========================================
# REQ #6: EVALUATION ENGINE
# ==========================================
import numpy as np
from tqdm import tqdm

def evaluate_system(retriever, chain, dataset):
    print("📊 Starting Evaluation...")

    # Metrics Storage
    retrieval_hits = 0      # HitRate@k
    mrr_score = 0           # Mean Reciprocal Rank
    gen_faithfulness = 0    # Simple Faithfulness Proxy
    total = len(dataset)

    results = []

    for item in tqdm(dataset):
        q = item["question"]
        expected_key = item["key_term"]

        # 1. TEST RETRIEVAL
        retrieved_docs = global_retriever.invoke(q)

        # Check if keyword exists in ANY retrieved doc (Hit Rate)
        hit = False
        rank = 0
        for i, doc in enumerate(retrieved_docs):
            if expected_key.lower() in doc.page_content.lower():
                hit = True
                rank = i + 1 # 1-based index
                break

        if hit:
            retrieval_hits += 1
            mrr_score += (1.0 / rank)

        # 2. TEST GENERATION
        generated_answer = chain.invoke(q)

        # Proxy Faithfulness: Does the answer contain the key term?
        # (In production, use Ragas or Arize Phoenix for this)
        faithful = expected_key.lower() in generated_answer.lower()
        if faithful:
            gen_faithfulness += 1

        results.append({
            "question": q,
            "hit": hit,
            "rank": rank,
            "faithful": faithful,
            "generated": generated_answer[:50] + "..."
        })

    # 3. CALCULATE FINAL SCORES
    print("\n🏆 FINAL EVALUATION REPORT")
    print("="*30)
    print(f"Hit Rate @ 3:    {retrieval_hits/total:.2%}  (Did we find the doc?)")
    print(f"MRR @ 3:         {mrr_score/total:.2f}    (Was it at the top?)")
    print(f"Faithfulness:    {gen_faithfulness/total:.2%}  (Did LLM use the info?)")
    print("="*30)

    return results

# Run the Eval
eval_results = evaluate_system(global_retriever, rag_chain, eval_dataset)

# 3.Future Recommendations
If this pipeline were to move to production, I would recommend:

* **Agentic Tool Use**: Leverage Qwen 2.5's tool-use capabilities to let the model decide when to search the paper versus when to use a calculator or Python REPL for math queries.

* **Re-Ranking:** Insert a Cross-Encoder (e.g., bge-reranker) after the Ensemble Retriever. This re-scores the top 10 documents deeply, improving MRR at the cost of slightly higher latency.

* **LLM-as-a-Judge:** Replace the keyword-based "Faithfulness" metric with a secondary LLM (using Ragas) to evaluate "Answer Relevance" and "Context Precision" more human-like.